# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
# Colab-compatible installation for this notebook
# Keep requests pinned to Colab's version to avoid dependency conflicts.
# SerpAPI package is required for the Search tool.

%pip uninstall -y chromadb langchain langchain-community langchain-openai langchain-experimental langchain-text-splitters langchain-classic > /dev/null 2>&1 || true

%pip install -q \
  "requests==2.32.4" \
  "langchain==0.3.23" \
  "langchain-community==0.3.21" \
  "langchain-experimental==0.3.4" \
  "langchain-openai==0.3.12" \
  "langchain-text-splitters==0.3.8" \
  "faiss-cpu==1.13.2" \
  "google-search-results>=2.4.2,<3" \
  "beautifulsoup4>=4.12,<5" \
  "tiktoken>=0.7,<1"


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 106.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages tha

In [2]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings


## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [3]:
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS


In [5]:
import os
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Required for the Search tool used later in the notebook.
SERPAPI_API_KEY = userdata.get("SERPAPI_API_KEY").strip()
os.environ["SERPAPI_API_KEY"] = SERPAPI_API_KEY


In [6]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [7]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)


/tmp/ipykernel_20120/3649600271.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)


In [8]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/tmp/ipykernel_20120/3064747660.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [9]:
OBJECTIVE = "Write a weather report for SF today"

In [10]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [11]:
baby_agi({"objective": OBJECTIVE})

/tmp/ipykernel_20120/3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should use the TODO action to create a todo list
Action: TODO
Action Input: "Create a todo list for writing a weather report for SF today"
Observation: 

1. Gather all necessary information: Start by collecting all the necessary information for writing a weather report for San Francisco today. This includes current temperature, humidity, wind speed, and any potential weather events.

2. Check reliable sources: Make sure to check reliable sources such as the National Weather Service or local news stations for accurate and up-to-date information.

3. Determine the format: Decide on the format for the weather report. Will it be a written report, a video, or a combination of both? This will help guide the rest of the process.

4. Create an outline: Before diving into writing, create an outline for the weather report. This will help organize the information 

{'objective': 'Write a weather report for SF today'}

In [12]:
# ===== UPDATED VECTOR STORE (fresh run for the new objective) =====

import faiss
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embeddings_model = OpenAIEmbeddings()

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)

updated_vectorstore = FAISS(
    embeddings_model.embed_query,
    index,
    InMemoryDocstore({}),
    {}
)

In [13]:
# ===== UPDATED PROMPTS + TOOLS + AGENT =====

from langchain_community.utilities import SerpAPIWrapper

# New planner prompt for a different objective
todo_prompt = PromptTemplate.from_template(
    """
You are an expert AI learning strategist and project planner.

Create a practical, well-structured todo list for this objective:
{objective}

Requirements:
- Break the objective into clear, actionable tasks
- Order tasks from research to execution to final deliverable
- Prefer tasks that produce concrete outputs
- Include at least one task for validation or review
"""
)

todo_chain = LLMChain(
    llm=OpenAI(temperature=0),
    prompt=todo_prompt
)

search = SerpAPIWrapper()

def calculator_tool(expression: str) -> str:
    """Useful for simple calculations such as time, budget, or scoring estimates."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Calculation error: {e}"

tools = [
    Tool(
        name="Search",
        func=search.run,
        description="Useful for finding recent or factual information on the web."
    ),
    Tool(
        name="Calculator",
        func=calculator_tool,
        description="Useful for doing simple calculations such as time estimates, totals, or comparisons."
    ),
]

prefix = """
You are an AI career coach and learning-plan execution agent.

Your job is to complete tasks that support the overall objective.
Use the available tools when they help you gather reliable information or make calculations.
When useful, summarize findings clearly and make the output practical.
"""

suffix = """
Task: {objective}
{agent_scratchpad}
"""

prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "agent_scratchpad"],
)

llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)

tool_names = [tool.name for tool in tools]

agent = ZeroShotAgent(
    llm_chain=llm_chain,
    allowed_tools=tool_names
)

agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent,
    tools=tools,
    verbose=True
)

In [14]:
# ===== UPDATED OBJECTIVE =====

OBJECTIVE = """
Create a 7-day beginner-friendly self-study plan for someone who wants to learn prompt engineering.

The final result should include:
- a day-by-day learning plan
- one recommended resource or activity per day
- estimated study time per day
- a final mini-project for day 7
"""

In [15]:
# ===== UPDATED BABYAGI =====

verbose = False
max_iterations = 4

baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=updated_vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [16]:
# ===== UPDATED RUN =====

baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Question: What is prompt engineering?
Thought: Before creating a learning plan, it's important to understand what prompt engineering is. 
Action: Search
Action Input: "Prompt Engineering"
Observation: ['Prompt engineering is a relatively new discipline for developing and optimizing prompts to efficiently use language models (LMs) for a wide variety of ...', 'Prompt engineering is the art and science of designing and optimizing prompts to guide AI models, particularly LLMs, towards generating the desired responses.', 'Prompt engineering is the process of writing, refining and optimizing inputs to encourage generative AI systems to create specific, high-quality outputs.', 'Prompt engineering is the process of structuring natural language inputs (known as prompts) to produce specified outputs from a generative artificial ...', 'With the OpenAI API, you can use a larg

{'objective': '\nCreate a 7-day beginner-friendly self-study plan for someone who wants to learn prompt engineering.\n\nThe final result should include:\n- a day-by-day learning plan\n- one recommended resource or activity per day\n- estimated study time per day\n- a final mini-project for day 7\n'}